# E-Commerce Demand Forecasting & Inventory Optimization

This portfolio notebook provides an inspection layer for the generated data, model governance, 2026 forecast, and inventory scenarios. The production-style logic lives in `Python/src/ecom_opt/`; the notebook intentionally reuses those persisted outputs instead of duplicating the pipeline.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
DATA = PROJECT_ROOT / 'Data'
plt.style.use('seaborn-v0_8-whitegrid')

## Historical Business Performance

In [ ]:
monthly = pd.read_csv(DATA / 'monthly_performance.csv')
monthly['YearMonth'] = pd.to_datetime(monthly['YearMonth'])
historical_summary = pd.Series({
    'Revenue TRY': monthly['NetRevenueTRY'].sum(),
    'Units sold': monthly['UnitsSold'].sum(),
    'Estimated demand': monthly['EstimatedDemandUnits'].sum(),
    'Lost sales units': monthly['LostSalesUnits'].sum(),
    'Demand fulfillment': monthly['UnitsSold'].sum() / monthly['EstimatedDemandUnits'].sum(),
    'Gross margin': monthly['GrossProfitTRY'].sum() / monthly['NetRevenueTRY'].sum(),
})
historical_summary

In [ ]:
ax = monthly.plot(x='YearMonth', y='NetRevenueTRY', figsize=(12, 4), color='#2F80ED', legend=False)
ax.set_title('Monthly Net Revenue | 2023–2025')
ax.set_ylabel('TRY')
ax.set_xlabel('')
plt.show()

## Model Governance

In [ ]:
models = pd.read_csv(DATA / 'model_comparison.csv')
model_summary = (
    models.groupby('Model')
    .agg(AverageWAPE=('WAPE', 'mean'), ChampionSKUs=('ChampionFlag', 'sum'))
    .sort_values('AverageWAPE')
)
model_summary.style.format({'AverageWAPE': '{:.1%}', 'ChampionSKUs': '{:.0f}'})

## 2026 Forecast

In [ ]:
forecast = pd.read_csv(DATA / 'forecast_monthly.csv')
forecast['YearMonth'] = pd.to_datetime(forecast['YearMonth'])
monthly_forecast = forecast.groupby('YearMonth', as_index=False)[['ForecastUnits', 'Lower80Units', 'Upper80Units']].sum()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly_forecast['YearMonth'], monthly_forecast['ForecastUnits'], color='#2F80ED', linewidth=2, label='Forecast')
ax.fill_between(monthly_forecast['YearMonth'], monthly_forecast['Lower80Units'], monthly_forecast['Upper80Units'], color='#56CCF2', alpha=.25, label='80% interval')
ax.set_title('2026 Monthly Demand Forecast')
ax.set_ylabel('Units')
ax.legend()
plt.show()

## Scenario and Replenishment Decisions

In [ ]:
scenarios = pd.read_csv(DATA / 'scenario_summary.csv').set_index('Scenario')
scenarios[['ForecastRevenueTRY', 'RecommendedInvestmentTRY', 'RecommendedOrderUnits', 'SKUsToOrder']].style.format({
    'ForecastRevenueTRY': 'TRY {:,.0f}',
    'RecommendedInvestmentTRY': 'TRY {:,.0f}',
    'RecommendedOrderUnits': '{:,.0f}',
    'SKUsToOrder': '{:,.0f}',
})

In [ ]:
replenishment = pd.read_csv(DATA / 'replenishment_recommendations.csv')
priority = replenishment.query("Action == 'ORDER NOW'").nlargest(10, 'RecommendedInvestmentTRY')
priority[['SKU', 'ProductName', 'Category', 'SupplierName', 'RecommendedOrderQty', 'RecommendedInvestmentTRY', 'ProjectedWeeksOfSupply']]

## Reproduce the Full Pipeline

Run from the repository root:

```bash
export PYTHONPATH=Python/src
python -m ecom_opt.run_pipeline
pytest -q
```

All data in this project is synthetic and intended for portfolio demonstration.